# AgentCore Harness — Managed Agent Loop

This notebook demonstrates:
- Creating a Harness (zero-code agent) with a single API call
- Invoking the harness and streaming responses
- Connecting tools (MCP servers, inline functions)
- Memory integration (short-term and long-term)
- Cost controls (maxIterations, timeoutSeconds, maxTokens)
- Direct shell execution on the harness environment

In [ ]:
# Install required packages
!pip install boto3 bedrock-agentcore -q

In [ ]:
import boto3
import json
import time
import uuid

REGION = "us-west-2"
ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]

# AgentCore clients
agentcore_control = boto3.client("bedrock-agentcore-control", region_name=REGION)
agentcore = boto3.client("bedrock-agentcore", region_name=REGION)
iam_client = boto3.client("iam")

print(f"Account: {ACCOUNT_ID}")
print(f"Region: {REGION}")

## 1. Create an Execution Role

The harness needs an IAM role it can assume. This role grants permissions for model invocation and any tools the agent will use.

In [ ]:
ROLE_NAME = "agentcore-harness-lab-role"

trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "bedrock-agentcore.amazonaws.com"},
        "Action": "sts:AssumeRole",
        "Condition": {"StringEquals": {"aws:SourceAccount": ACCOUNT_ID}}
    }]
}

permissions_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": ["bedrock:InvokeModel", "bedrock:InvokeModelWithResponseStream"],
            "Resource": "*"
        },
        {
            "Effect": "Allow",
            "Action": ["bedrock-agentcore:*"],
            "Resource": "*"
        }
    ]
}

try:
    role_response = iam_client.create_role(
        RoleName=ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(trust_policy)
    )
    iam_client.put_role_policy(
        RoleName=ROLE_NAME,
        PolicyName="harness-permissions",
        PolicyDocument=json.dumps(permissions_policy)
    )
    ROLE_ARN = role_response["Role"]["Arn"]
    print(f"✓ Created role: {ROLE_ARN}")
    time.sleep(10)  # Wait for propagation
except iam_client.exceptions.EntityAlreadyExistsException:
    ROLE_ARN = iam_client.get_role(RoleName=ROLE_NAME)["Role"]["Arn"]
    print(f"✓ Role exists: {ROLE_ARN}")

## 2. Create a Harness

A harness is defined by:
- **Execution role** — what the agent can access
- **System prompt** (optional) — agent persona and instructions
- **Model** (optional) — defaults to Claude Sonnet on Bedrock

No orchestration code needed. AgentCore manages the full ReAct loop.

In [ ]:
HARNESS_NAME = "lab-research-agent"

response = agentcore_control.create_harness(
    harnessName=HARNESS_NAME,
    executionRoleArn=ROLE_ARN,
    systemPrompt=[{"text": "You are a helpful research assistant. Be concise and factual."}],
    modelId="us.amazon.nova-2-lite-v1:0"
)

HARNESS_ID = response["harnessId"]
HARNESS_ARN = response["arn"]
print(f"✓ Created harness: {HARNESS_ID}")
print(f"  ARN: {HARNESS_ARN}")

In [ ]:
# Wait for harness to be ready
print("Waiting for harness to become READY...")
for _ in range(30):
    status = agentcore_control.get_harness(harnessId=HARNESS_ID)
    if status["status"] == "READY":
        print(f"✓ Harness is READY")
        break
    time.sleep(5)
else:
    print(f"⚠ Harness status: {status['status']}")

## 3. Invoke the Harness (Streaming)

The `invoke_harness` API returns a stream of events. The agent reasons, calls tools, and generates responses — all managed by Harness.

**Key rule**: `runtimeSessionId` must be at least 33 characters. Reuse the same ID to continue a conversation.

In [ ]:
def invoke_harness(messages, tools=None, session_id=None, **kwargs):
    """Invoke the harness and collect streamed response text."""
    session_id = session_id or str(uuid.uuid4()) + "-" + str(uuid.uuid4())[:8]
    
    params = {
        "harnessArn": HARNESS_ARN,
        "runtimeSessionId": session_id,
        "messages": messages,
    }
    if tools:
        params["tools"] = tools
    params.update(kwargs)
    
    response = agentcore.invoke_harness(**params)
    
    full_text = ""
    tool_calls = []
    stop_reason = None
    usage = {}
    
    for event in response["stream"]:
        if "contentBlockDelta" in event:
            delta = event["contentBlockDelta"].get("delta", {})
            if "text" in delta:
                print(delta["text"], end="", flush=True)
                full_text += delta["text"]
        elif "contentBlockStart" in event:
            start = event["contentBlockStart"].get("start", {})
            if "toolUse" in start:
                tool_calls.append(start["toolUse"])
        elif "messageStop" in event:
            stop_reason = event["messageStop"].get("stopReason")
        elif "metadata" in event:
            usage = event["metadata"].get("usage", {})
        elif "runtimeClientError" in event:
            print(f"\nError: {event['runtimeClientError']['message']}")
    
    print()  # newline
    return {"text": full_text, "stop_reason": stop_reason, "usage": usage, "tool_calls": tool_calls, "session_id": session_id}

In [ ]:
# Simple invocation
result = invoke_harness(
    messages=[{"role": "user", "content": [{"text": "What is Amazon Bedrock AgentCore in 2 sentences?"}]}]
)
print(f"\nStop reason: {result['stop_reason']}")
print(f"Usage: {result['usage']}")

## 4. Multi-Turn Conversation (Same Session)

Reuse the same `runtimeSessionId` to continue the conversation. The harness maintains context in the microVM.

In [ ]:
# Multi-turn: same session
SESSION_ID = str(uuid.uuid4()) + "-" + str(uuid.uuid4())[:8]

print("--- Turn 1 ---")
result1 = invoke_harness(
    messages=[{"role": "user", "content": [{"text": "My name is Alice. I'm researching serverless architectures."}]}],
    session_id=SESSION_ID
)

print("\n--- Turn 2 ---")
result2 = invoke_harness(
    messages=[{"role": "user", "content": [{"text": "What's my name and what am I researching?"}]}],
    session_id=SESSION_ID
)
# The agent remembers context from turn 1

## 5. Connect Tools — Inline Functions

Inline functions execute **client-side** (your code), not on the harness VM. When the agent calls the tool, the stream pauses with `stopReason: "tool_use"` — you execute the function and send the result back.

This is the pattern for:
- Human-in-the-loop approvals
- Calling internal APIs
- Custom business logic

In [ ]:
# Define an inline function tool
weather_tool = {
    "type": "inline_function",
    "name": "get_weather",
    "config": {
        "inlineFunction": {
            "description": "Get current weather for a city.",
            "inputSchema": {
                "type": "object",
                "properties": {"city": {"type": "string", "description": "City name"}},
                "required": ["city"]
            }
        }
    }
}

# Simulated weather function (your business logic)
def get_weather(city):
    """Simulate a weather API call."""
    data = {
        "Seattle": {"temp": "58°F", "condition": "Rainy"},
        "Miami": {"temp": "85°F", "condition": "Sunny"},
        "New York": {"temp": "72°F", "condition": "Partly Cloudy"},
    }
    return data.get(city, {"temp": "Unknown", "condition": "No data"})

print("Tool defined. Next: invoke and handle the tool_use loop.")

In [ ]:
# Invoke with inline function — full tool_use loop
session_id = str(uuid.uuid4()) + "-" + str(uuid.uuid4())[:8]

# Step 1: Initial invocation
response = agentcore.invoke_harness(
    harnessArn=HARNESS_ARN,
    runtimeSessionId=session_id,
    tools=[weather_tool],
    messages=[{"role": "user", "content": [{"text": "What's the weather in Seattle?"}]}],
)

# Step 2: Process the stream, capture tool_use
tool_use_id = None
tool_input = ""
assistant_content = []

for event in response["stream"]:
    if "contentBlockStart" in event:
        start = event["contentBlockStart"].get("start", {})
        if "toolUse" in start:
            tool_use_id = start["toolUse"]["toolUseId"]
            print(f"Agent calling tool: {start['toolUse']['name']}")
    elif "contentBlockDelta" in event:
        delta = event["contentBlockDelta"].get("delta", {})
        if "text" in delta:
            print(delta["text"], end="", flush=True)
        elif "toolUse" in delta:
            tool_input += delta["toolUse"].get("input", "")
    elif "messageStop" in event:
        stop = event["messageStop"]["stopReason"]
        print(f"\nStop reason: {stop}")

print(f"Tool call ID: {tool_use_id}")
print(f"Tool input: {tool_input}")

In [ ]:
# Step 3: Execute the tool locally and send result back
if tool_use_id:
    tool_args = json.loads(tool_input) if tool_input else {"city": "Seattle"}
    weather_result = get_weather(tool_args.get("city", "Seattle"))
    print(f"Local execution result: {weather_result}")
    
    # Send tool result back to the harness
    response = agentcore.invoke_harness(
        harnessArn=HARNESS_ARN,
        runtimeSessionId=session_id,
        tools=[weather_tool],
        messages=[
            {"role": "assistant", "content": [{"toolUse": {"toolUseId": tool_use_id, "name": "get_weather", "input": tool_args}}]},
            {"role": "user", "content": [{"toolResult": {"toolUseId": tool_use_id, "content": [{"text": json.dumps(weather_result)}]}}]}
        ],
    )
    
    # Read final response
    print("\nAgent response:")
    for event in response["stream"]:
        if "contentBlockDelta" in event:
            delta = event["contentBlockDelta"].get("delta", {})
            if "text" in delta:
                print(delta["text"], end="", flush=True)
    print()

## 6. Connect Remote MCP Server

The harness can connect to any remote MCP server by URL. Tools are declarative — AgentCore handles invocation and credentials.

In [ ]:
# Invoke with a remote MCP server tool
mcp_tools = [
    {
        "type": "remote_mcp",
        "name": "time-server",
        "config": {"remoteMcp": {"url": "https://mcp.example.com/time"}},
    }
]

# Note: This is a demonstration of the API shape.
# Replace with a real MCP server URL to test.
print("MCP tool configuration:")
print(json.dumps(mcp_tools, indent=2))
print("\nTo invoke: pass tools=mcp_tools to invoke_harness()")

## 7. Memory Integration

Attach AgentCore Memory to the harness for:
- **Short-term**: Raw conversation events within a session
- **Long-term**: Extracted knowledge across sessions (semantic, summarization, user preference)
- **Actor scoping**: Isolate memory per user with `actorId`

In [ ]:
# Create a Memory instance
memory_response = agentcore_control.create_memory(
    name="lab-harness-memory",
    description="Memory for harness lab",
    eventExpiryDuration=30  # days
)

MEMORY_ARN = memory_response["arn"]
MEMORY_ID = memory_response["memoryId"]
print(f"✓ Created memory: {MEMORY_ID}")
print(f"  ARN: {MEMORY_ARN}")

In [ ]:
# Attach memory to the harness
agentcore_control.update_harness(
    harnessId=HARNESS_ID,
    memory={"optionalValue": {
        "agentCoreMemoryConfiguration": {"arn": MEMORY_ARN}
    }}
)
print(f"✓ Attached memory to harness")

# Now the harness auto-persists conversations.
# On subsequent invocations with the same session ID,
# history is loaded from Memory automatically.

In [ ]:
# Demonstrate actor-scoped memory
print("--- Session with actor 'alice' ---")
result = invoke_harness(
    messages=[{"role": "user", "content": [{"text": "Remember that my favorite language is Python."}]}],
    actorId="alice"
)

print("\n--- New session, same actor 'alice' ---")
result = invoke_harness(
    messages=[{"role": "user", "content": [{"text": "What's my favorite programming language?"}]}],
    actorId="alice"
)
# Long-term memory carries over even in a new session

## 8. Cost Controls

Prevent runaway agents with hard limits:

| Parameter | Default | Purpose |
|-----------|---------|--------|
| `maxIterations` | 75 | Max reasoning/action cycles |
| `timeoutSeconds` | 3600 | Wall-clock timeout |
| `maxTokens` | N/A | Token budget per invocation |
| `idleRuntimeSessionTimeout` | 900 | Idle microVM lifetime (seconds) |
| `maxLifetime` | 28800 | Max microVM lifetime (seconds) |

In [ ]:
# Update harness with cost controls
agentcore_control.update_harness(
    harnessId=HARNESS_ID,
    maxIterations=20,
    timeoutSeconds=120,
    maxTokens=4096
)
print("✓ Updated cost controls:")
print("  maxIterations: 20")
print("  timeoutSeconds: 120")
print("  maxTokens: 4096")

In [ ]:
# Override limits on a single invocation
result = invoke_harness(
    messages=[{"role": "user", "content": [{"text": "Write a haiku about cloud computing."}]}],
    maxIterations=5,
    timeoutSeconds=30
)
print(f"Stop reason: {result['stop_reason']}")

## 9. Direct Shell Execution

`InvokeAgentRuntimeCommand` gives direct shell access to the microVM **without going through the model**. No token cost, no reasoning — just deterministic command execution.

Use cases:
- Install dependencies before an invocation
- Inspect files the agent created
- Run tests or validation scripts

In [ ]:
# Execute a command directly on the harness environment
session_id = str(uuid.uuid4()) + "-" + str(uuid.uuid4())[:8]

response = agentcore.invoke_agent_runtime_command(
    agentRuntimeArn=HARNESS_ARN,
    runtimeSessionId=session_id,
    body={"command": "echo 'Hello from harness VM' && python3 --version && ls /"}
)

print("Shell output:")
for event in response["stream"]:
    chunk = event.get("chunk", {})
    if "contentDelta" in chunk:
        delta = chunk["contentDelta"]
        if "stdout" in delta:
            print(delta["stdout"], end="", flush=True)
        if "stderr" in delta:
            print(f"[stderr] {delta['stderr']}", end="", flush=True)
    elif "contentStop" in chunk:
        print(f"\n[exit code: {chunk['contentStop']['exitCode']}]")

## 10. Harness Configuration Summary

View the full harness configuration.

In [ ]:
# Get current harness config
config = agentcore_control.get_harness(harnessId=HARNESS_ID)

print("Harness Configuration:")
print("=" * 60)
print(f"  Name: {config.get('harnessName')}")
print(f"  ID: {config.get('harnessId')}")
print(f"  Status: {config.get('status')}")
print(f"  Model: {config.get('modelId', 'default')}")
print(f"  Max Iterations: {config.get('maxIterations')}")
print(f"  Timeout: {config.get('timeoutSeconds')}s")
print(f"  Max Tokens: {config.get('maxTokens')}")
print(f"  Memory: {config.get('memory', 'None')}")
print(f"  Created: {config.get('createdAt')}")

## 11. Cleanup

In [ ]:
# Delete harness and memory
try:
    agentcore_control.delete_harness(harnessId=HARNESS_ID)
    print(f"✓ Deleted harness: {HARNESS_ID}")
except Exception as e:
    print(f"Harness cleanup: {e}")

try:
    agentcore_control.delete_memory(memoryId=MEMORY_ID)
    print(f"✓ Deleted memory: {MEMORY_ID}")
except Exception as e:
    print(f"Memory cleanup: {e}")

try:
    iam_client.delete_role_policy(RoleName=ROLE_NAME, PolicyName="harness-permissions")
    iam_client.delete_role(RoleName=ROLE_NAME)
    print(f"✓ Deleted IAM role: {ROLE_NAME}")
except Exception as e:
    print(f"IAM cleanup: {e}")

print("\n✓ Cleanup complete")

## Summary

| Feature | API |
|---------|-----|
| Create agent | `create_harness(name, role, systemPrompt, modelId)` |
| Invoke (streaming) | `invoke_harness(harnessArn, runtimeSessionId, messages)` |
| Add tools | Pass `tools=[]` at create/update/invoke time |
| Inline functions | Tool executes client-side, send result back via messages |
| Memory | `update_harness(memory={agentCoreMemoryConfiguration})` |
| Cost controls | `maxIterations`, `timeoutSeconds`, `maxTokens` |
| Shell access | `invoke_agent_runtime_command(command=...)` |

**Key differences from AgentCore Runtime:**
- No container to build or push
- No orchestration code (Strands, LangGraph, etc.)
- Config-based: model + prompt + tools = working agent
- When you need full control, export to Strands-based code